## What is Pydantic
#### Pydantic is a data validation and settings management library using Python type annotations.

It’s built around the concept of defining schemas as Python classes. These schemas ensure the data matches the expected types, and optionally transforms or validates the data automatically.

### BaseModel

In [1]:
from pydantic import BaseModel

class Book(BaseModel):
    title: str
    author: str
    pages : int
    published: bool = True # Default value

book = Book(title="The Great Gatsby", author="F. Scott Fitzgerald", pages=180)
print(book)

title='The Great Gatsby' author='F. Scott Fitzgerald' pages=180 published=True


### Validation and type coersion

In [4]:
book = Book(title="1984", author="George Orwell", pages=328, published=False)
print(book.pages) # int : 328 , even though input was a string

328


In [7]:
## if somthing doesn't really match
Book(title="The Great Gatsby", author="F. Scott Fitzgerald", pages="not_a_number") # will throw an error because pages is not an int
# raises validation error!

ValidationError: 1 validation error for Book
pages
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='not_a_number', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/int_parsing

### Instantiating with **fields

In [8]:
fields = {
    "title": "The Great Gatsby",
    "author": "F. Scott Fitzgerald",
    "pages": 180,
}

book = Book(**fields)  # Unpacking the dictionary into keyword arguments
print(book)

title='The Great Gatsby' author='F. Scott Fitzgerald' pages=180 published=True


### From and TO JSON

In [9]:
print(book.model_dump())

{'title': 'The Great Gatsby', 'author': 'F. Scott Fitzgerald', 'pages': 180, 'published': True}


In [10]:
print(book.model_dump_json())

{"title":"The Great Gatsby","author":"F. Scott Fitzgerald","pages":180,"published":true}


### using standard json module

In [11]:
import json
json_string = json.dumps(book.model_dump())
print(json_string)  # This will print the JSON string representation of the book object

{"title": "The Great Gatsby", "author": "F. Scott Fitzgerald", "pages": 180, "published": true}


### Loading from json

In [12]:
json_string = '{"title": "The Great Gatsby", "author": "F. Scott Fitzgerald", "pages": 180, "published": true}'
data = json.loads(json_string)
book = Book(**data)  # Unpacking the dictionary into keyword arguments
print(book)

title='The Great Gatsby' author='F. Scott Fitzgerald' pages=180 published=True


## Nested Models

In [13]:
class Author(BaseModel):
    name: str
    birth_year: int

class Book(BaseModel):
    title: str
    author: Author
    pages: int

a = Author(name="F. Scott Fitzgerald", birth_year=1896)
b = Book(title="The Great Gatsby", author=a, pages=180)
print(b)

title='The Great Gatsby' author=Author(name='F. Scott Fitzgerald', birth_year=1896) pages=180


### Build from nested dicts

In [14]:
data = {
    "title": "The Great Gatsby",
    "author": {
        "name": "F. Scott Fitzgerald",
        "birth_year": 1896
    },
    "pages": 180
}

b = Book(**data)
print(b)

title='The Great Gatsby' author=Author(name='F. Scott Fitzgerald', birth_year=1896) pages=180


## Exercise
#### Create a movie model
##### Define a model with fields:

###### title (str)
###### year (int)
###### genres (list of str)
###### duration_minutes (int)
###### rating (float, default to 5.0)
###### Instantiate it using a dictionary

In [27]:
from pydantic import BaseModel, Field
class Director(BaseModel):
    name: str
    born: int


class Movie(BaseModel):
    title: str = Field(..., max_length=100)  # Required field with max length
    year: int
    genres: list[str]
    duration_minutes: int
    rating: float = Field(..., ge=0.0, le=10.0)  # Rating between 0 and 10
    director: Director | None = None  # Optional field

    def summarize(self) -> str:
        return f"{self.title} ({self.year}) - {', '.join(self.genres)}"


movie = {
    "title": "Inception",
    "year": 2010,
    "genres": ["Action", "Adventure", "Sci-Fi"],
    "duration_minutes": 148,
    "rating": 8.8,
    "director": {
        "name": "Christopher Nolan",
        "born": 1970
    }
}

movie = Movie(**movie)
print(movie)
print(movie.summarize())

title='Inception' year=2010 genres=['Action', 'Adventure', 'Sci-Fi'] duration_minutes=148 rating=8.8 director=Director(name='Christopher Nolan', born=1970)
Inception (2010) - Action, Adventure, Sci-Fi
